In [1]:
import sys
import torch
import platform

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

!nvidia-smi

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: Tesla T4
Tue May  5 14:23:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       3MiB /  15360MiB |      0%     

In [2]:
import os
import torch
import sys
import platform

# 1. Clone Repository (Safe to re-run)
if not os.path.exists("InfiniteTalk"):
    !git clone https://github.com/MeiGen-AI/InfiniteTalk
else:
    print("Repository already cloned.")

# 2. Enter Directory (Required after every restart)
%cd InfiniteTalk

# 3. Safe Dependency Installation
py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_ver = torch.__version__.split("+")[0]
torch_ver_short = ".".join(torch_ver.split(".")[:2])
cuda_ver = torch.version.cuda.replace(".", "")[:3] if torch.version.cuda else "cpu"
abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"

print(f"✅ Detected: Python={py_ver}, Torch={torch_ver}, CUDA={cuda_ver}, ABI={abi}")
print("Installing dependencies... (Fast mode enabled)")

# Install dependencies needed for extensions
!pip install --no-cache-dir packaging ninja psutil wheel

# Install Flash Attention 2 (Dynamic Wheel Selection)
try:
    import flash_attn
    print("✅ Flash Attention already installed.")
except ImportError:
    print("Installing Flash Attention...")

    # CASE 1: Python 3.12 + PyTorch 2.9 (User Verified Environment)
    if py_ver == "cp312" and torch_ver.startswith("2.9"):
        url = "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
    else:
        # CASE 2: Fallback to guessing official URL for other versions
        fa_v = "2.8.3"
        cu_tag = f"cu{cuda_ver[:2]}" # e.g., cu12
        wheel = f"flash_attn-{fa_v}+{cu_tag}torch{torch_ver_short}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl"
        url = f"https://github.com/Dao-AILab/flash-attention/releases/download/v{fa_v}/{wheel}"

    try:
        print(f"Attempting to install: {url}")
        res = os.system(f"pip install --no-cache-dir {url}")
        if res != 0:
            raise Exception("Pip install returned non-zero exit code")
        print(f"✅ Successfully installed Flash Attention via wheel")
    except Exception as e:
        print(f"⚠️ Wheel install failed: {e}")
        print("Falling back to standard pip install (this may trigger a slow build)...")
        !pip install flash-attn --no-build-isolation

# Install other dependencies from requirements.txt
!sed -i 's/numpy>=1.23.5,<2/numpy>=1.23.5/' requirements.txt
!pip install --no-cache-dir -r requirements.txt
!pip install --no-cache-dir librosa huggingface_hub

# Install system dependencies
!apt-get update && apt-get install -y ffmpeg > /dev/null 2>&1

print("\n✅ Setup Complete! If you see a 'Restart Session' button, you can IGNORE it and try running the next cell.")

Cloning into 'InfiniteTalk'...
remote: Enumerating objects: 513, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 513 (delta 120), reused 80 (delta 80), pack-reused 353 (from 1)
Receiving objects: 100% (513/513), 241.41 MiB | 29.53 MiB/s, done.
Resolving deltas: 100% (187/187), done.
/content/InfiniteTalk
✅ Detected: Python=cp312, Torch=2.10.0, CUDA=128, ABI=TRUE
Installing dependencies... (Fast mode enabled)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 10.2 MB/s eta 0:00:00
Installing Flash Attention...
Attempting to install: https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
⚠️ Wheel install failed: Pip install returned non-zero exit code
Falling back to standard pip install (this may trigger a slow build)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 89.3 MB/s eta 0:00:00
  Preparing metadata (setup

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,608 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,601 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/

In [3]:
import os
os.makedirs("weights", exist_ok=True)
os.makedirs("weights/InfiniteTalk/quant_models", exist_ok=True)

# PREVENT DOUBLE DISK USAGE: Set HF cache to weights folder
os.environ["HF_HUB_CACHE"] = os.path.join(os.getcwd(), "weights/.cache")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "true"

print("Downloading models (Optimized for space)...")

# 1. Download base model (Wan2.1-I2V-14B-480P)
!hf download Wan-AI/Wan2.1-I2V-14B-480P --local-dir ./weights/Wan2.1-I2V-14B-480P

# 2. Download FP8 Quantized Weights (SAVED DISK & VRAM)
# These are smaller and required for Tesla T4 (16GB VRAM)
print("Downloading FP8 weights for Tesla T4 compatibility...")
#!wget -O weights/InfiniteTalk/quant_models/infinitetalk_single_fp8.safetensors https://huggingface.co/MeiGen-AI/InfiniteTalk/resolve/main/quant_models/infinitetalk_single_fp8.safetensors
#!wget -O weights/InfiniteTalk/quant_models/infinitetalk_multi_fp8.safetensors https://huggingface.co/MeiGen-AI/InfiniteTalk/resolve/main/quant_models/infinitetalk_multi_fp8.safetensors
!wget -O weights/InfiniteTalk/quant_models/t5_fp8.safetensors https://huggingface.co/MeiGen-AI/InfiniteTalk/resolve/main/quant_models/t5_fp8.safetensors
!wget -O weights/InfiniteTalk/quant_models/t5_map_fp8.json https://huggingface.co/MeiGen-AI/InfiniteTalk/resolve/main/quant_models/t5_map_fp8.json

# 3. Download English audio encoder
!hf download facebook/wav2vec2-base-960h --local-dir ./weights/wav2vec2-base-960h

print("✅ Download Complete!")


  A new version of huggingface_hub is available: 1.11.0 → 1.13.0

  Do you want to update now? [Y/n] (/usr/bin/python3 -m pip install -U huggingface_hub) Y

  Running: /usr/bin/python3 -m pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 16.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0

  ✓ Successfully updated huggingface_hub to 1.13.0. Please re-run your command.
--2026-05-05 14:30:20--  https://huggingface.co/MeiGen-AI/InfiniteTalk/resolve/main/quant_models/t5_fp8.safetensors
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.118, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/68a2e834fdfab51069736146/52144

In [4]:
# FIX

!pip install --force-reinstall torch torchvision torchaudio xformers

!sed -i 's/from inspect import ArgSpec/from inspect import FullArgSpec as ArgSpec/g' /content/InfiniteTalk/wan/multitalk.py

!pip install misaki[en]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of spacy-curated-transformers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 54.6 MB/s eta 0:00:

In [ ]:
!hf download Wan-AI/Wan2.1-I2V-14B-480P --local-dir ./weights/Wan2.1-I2V-14B-480P

Fetching 33 files:   0% 0/33 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

In [5]:
# Enable public link for Gradio

!sed -i 's/demo.launch(server_name="0.0.0.0", debug=True, server_port=8418)/demo.launch(server_name="0.0.0.0", debug=True, server_port=8418, share=True)/' app.py

print("🚀 Launching in FP8 Mode (Compatible with Tesla T4 16GB)")

# Launch Gradio interface with FP8 Optimization enabled
!python app.py \
    --ckpt_dir weights/Wan2.1-I2V-14B-480P \
    --wav2vec_dir 'weights/wav2vec2-base-960h' \
    --infinitetalk_dir weights/InfiniteTalk/single/infinitetalk.safetensors \
    --quant fp8 \
    --quant_dir weights/InfiniteTalk/quant_models/infinitetalk_single_fp8.safetensors \
    --num_persistent_param_in_dit 0 \
    --motion_frame 9
#/content/InfiniteTalk/weights/InfiniteTalk/quant_models/t5_fp8.safetensors

🚀 Launching in FP8 Mode (Compatible with Tesla T4 16GB)
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading weights: 100% 210/210 [00:00<00:00, 613.97it/s, Materializing param=feature_projection.projection.weight]
Wav2Vec2Model LOAD REPORT from: weights/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Traceback (most recent call last):
  File "/usr/local/l